# Solara AI — Synthetic Shading Generation

> **DISCLOSURE:** The Shading class contains synthetically generated images (simulated shadow overlays on clean panel photos), as no source dataset provided real shading examples. This should be disclosed in the project report and ideally supplemented with real images if available before final submission.

**Goal:** Generate 300-400 synthetic shading images by applying realistic shadow overlays to clean solar panel images.

**Techniques:**
- Soft-edged dark polygon overlays (branch/pole shadows)
- Linear gradient darkening (building shadow edges)
- Gaussian blur on all shadow edges
- Random shadow darkness (20-60%) and size (15-40% of panel)

In [ ]:
import os
import sys
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 0

random.seed(42)
np.random.seed(42)

# Paths
_notebook_dir = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
_model_training_dir = _notebook_dir.parent
RAW_DIR = _model_training_dir / "dataset_raw"
OUTPUT_DIR = RAW_DIR / "synthetic_shading" / "shading"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.tif'}
TARGET_COUNT = 350  # Target number of synthetic images

print(f"Output directory: {OUTPUT_DIR}")
print(f"Target synthetic images: {TARGET_COUNT}")

## 1. Collect Clean Source Images

Gather all images from the "clean" reference bucket across all sources. These are images NOT assigned to any fault class.

In [ ]:
def collect_clean_images():
    """Collect all clean images from all sources."""
    clean_paths = []
    
    # source_2_dust/Clean (1493 images)
    s2_clean = RAW_DIR / "source_2_dust" / "Clean"
    if s2_clean.exists():
        paths = [p for p in s2_clean.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
        clean_paths.extend(paths)
        print(f"  source_2_dust/Clean: {len(paths)} images")
    
    # source_1_afroz/Clean (all splits)
    for split in ['train', 'val', 'test']:
        s1_split = RAW_DIR / "source_1_afroz" / split / "Clean"
        if s1_split.exists():
            paths = [p for p in s1_split.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
            clean_paths.extend(paths)
            print(f"  source_1_afroz/{split}/Clean: {len(paths)} images")
    
    # source_1_afroz top-level Clean
    s1_top = RAW_DIR / "source_1_afroz" / "Clean"
    if s1_top.exists():
        paths = [p for p in s1_top.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
        clean_paths.extend(paths)
        print(f"  source_1_afroz/Clean (top): {len(paths)} images")
    
    # source_3_pv_defect/Clean (all splits)
    for split in ['train', 'val', 'test']:
        s3_split = RAW_DIR / "source_3_pv_defect" / split / "Clean"
        if s3_split.exists():
            paths = [p for p in s3_split.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
            clean_paths.extend(paths)
            print(f"  source_3_pv_defect/{split}/Clean: {len(paths)} images")
    
    return clean_paths

print("Collecting clean source images...")
clean_images = collect_clean_images()
print(f"\nTotal clean images available: {len(clean_images)}")
print(f"Need to generate: {TARGET_COUNT} synthetic shading images")
print(f"Each base image will get 1-3 shadow variations")

## 2. Synthetic Shadow Generation

Three shadow simulation techniques, randomly chosen per image:
1. **Polygon shadow** — soft-edged dark polygon (branch/pole shadow)
2. **Gradient shadow** — linear gradient darkening (building shadow edge)
3. **Radial shadow** — darkening from one corner/edge

In [ ]:
def create_soft_mask(size, polygon_points, blur_radius=25):
    """Create a soft-edged binary mask from a polygon."""
    mask = Image.new('L', size, 0)
    draw = ImageDraw.Draw(mask)
    draw.polygon(polygon_points, fill=255)
    # Gaussian blur for soft edges
    mask = mask.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    return mask

def apply_shadow_multiply(img_array, shadow_mask, darkness):
    """Apply shadow using multiply blend (darkens the image)."""
    shadow_layer = np.full_like(img_array, 255 - int(255 * darkness), dtype=np.float32)
    mask_normalized = np.array(shadow_mask, dtype=np.float32) / 255.0
    
    # Interpolate between original and darkened
    result = img_array.astype(np.float32)
    for c in range(3):
        result[:, :, c] = img_array[:, :, c] * (1 - mask_normalized * darkness)
    
    return np.clip(result, 0, 255).astype(np.uint8)

def generate_polygon_shadow(img_w, img_h):
    """Generate a random polygon shadow (branch/pole simulation)."""
    # Random polygon with 4-8 vertices
    num_vertices = random.randint(4, 8)
    
    # Random center position
    cx = random.randint(img_w // 4, 3 * img_w // 4)
    cy = random.randint(img_h // 4, 3 * img_h // 4)
    
    # Random radius (15-40% of image diagonal)
    diag = np.sqrt(img_w**2 + img_h**2)
    radius = random.uniform(0.15, 0.40) * diag
    
    # Generate polygon points around center with random jitter
    angles = np.sort(np.random.uniform(0, 2 * np.pi, num_vertices))
    points = []
    for angle in angles:
        r = radius * random.uniform(0.5, 1.5)
        px = cx + r * np.cos(angle)
        py = cy + r * np.sin(angle)
        points.append((int(np.clip(px, 0, img_w)), int(np.clip(py, 0, img_h))))
    
    return points

def generate_gradient_shadow(img_w, img_h):
    """Generate a linear gradient shadow mask."""
    # Random angle (0-360 degrees)
    angle = random.uniform(0, 360)
    
    # Create gradient
    Y, X = np.mgrid[0:img_h, 0:img_w]
    rad = np.radians(angle)
    
    # Project positions onto gradient axis
    projection = X * np.cos(rad) + Y * np.sin(rad)
    proj_min = projection.min()
    proj_max = projection.max()
    
    # Normalize to 0-1
    gradient = (projection - proj_min) / (proj_max - proj_min + 1e-6)
    
    # Apply non-linear curve for more realistic shadow edge
    gradient = np.power(gradient, random.uniform(0.5, 2.0))
    
    # Convert to PIL Image
    gradient_uint8 = (gradient * 255).astype(np.uint8)
    mask = Image.fromarray(gradient_uint8, mode='L')
    
    return mask

def generate_radial_shadow(img_w, img_h):
    """Generate a radial shadow from a random edge/corner."""
    # Random origin point (on edges)
    side = random.choice(['top', 'bottom', 'left', 'right', 'tl', 'tr', 'bl', 'br'])
    if side == 'top':
        ox, oy = random.randint(0, img_w), 0
    elif side == 'bottom':
        ox, oy = random.randint(0, img_w), img_h
    elif side == 'left':
        ox, oy = 0, random.randint(0, img_h)
    elif side == 'right':
        ox, oy = img_w, random.randint(0, img_h)
    elif side == 'tl':
        ox, oy = 0, 0
    elif side == 'tr':
        ox, oy = img_w, 0
    elif side == 'bl':
        ox, oy = 0, img_h
    else:
        ox, oy = img_w, img_h
    
    Y, X = np.mgrid[0:img_h, 0:img_w]
    dist = np.sqrt((X - ox)**2 + (Y - oy)**2)
    max_dist = np.sqrt(img_w**2 + img_h**2)
    
    gradient = dist / max_dist
    gradient = np.clip(gradient, 0, 1)
    
    # Invert so shadow is near the origin
    gradient = 1 - gradient
    
    gradient_uint8 = (gradient * 255).astype(np.uint8)
    mask = Image.fromarray(gradient_uint8, mode='L')
    
    return mask

print("Shadow generation functions defined.")

In [ ]:
def apply_synthetic_shadow(img_path, output_path):
    """Apply a random synthetic shadow to an image and save."""
    # Load image
    img = Image.open(img_path).convert('RGB')
    img_array = np.array(img)
    h, w = img_array.shape[:2]
    
    # Random shadow parameters
    darkness = random.uniform(0.20, 0.60)
    blur_radius = random.randint(15, 40)
    
    # Choose shadow type randomly
    shadow_type = random.choice(['polygon', 'gradient', 'radial'])
    
    if shadow_type == 'polygon':
        points = generate_polygon_shadow(w, h)
        mask = create_soft_mask((w, h), points, blur_radius)
    elif shadow_type == 'gradient':
        mask = generate_gradient_shadow(w, h)
        mask = mask.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    else:  # radial
        mask = generate_radial_shadow(w, h)
        mask = mask.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    
    # Apply shadow using multiply blend
    result = apply_shadow_multiply(img_array, mask, darkness)
    
    # Save
    result_img = Image.fromarray(result)
    result_img.save(output_path, quality=92)
    
    return shadow_type, darkness

# Generate synthetic shading images
print(f"Generating {TARGET_COUNT} synthetic shading images...")
print(f"Using {len(clean_images)} clean base images")
print()

generated = 0
output_paths = []

# We may need multiple variations per base image
base_image_queue = list(clean_images)
random.shuffle(base_image_queue)
queue_idx = 0

while generated < TARGET_COUNT:
    # Get next base image (cycle if needed)
    if queue_idx >= len(base_image_queue):
        queue_idx = 0
        random.shuffle(base_image_queue)
    
    base_path = base_image_queue[queue_idx]
    queue_idx += 1
    
    # Generate output filename
    output_name = f"synthetic_shading_{generated + 1:04d}.jpg"
    output_path = OUTPUT_DIR / output_name
    
    try:
        shadow_type, darkness = apply_synthetic_shadow(base_path, output_path)
        output_paths.append(output_path)
        generated += 1
        
        if generated % 50 == 0:
            print(f"  Generated {generated}/{TARGET_COUNT}...")
    except Exception as e:
        print(f"  Skipping {base_path.name}: {e}")
        continue

print(f"\nDone! Generated {generated} synthetic shading images")
print(f"Saved to: {OUTPUT_DIR}")

## 3. Visual Validation

Grid of 12 generated synthetic shading examples to confirm they look like plausible real shadows.

In [ ]:
# Plot 12 random samples
sample_paths = random.sample(output_paths, min(12, len(output_paths)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Synthetic Shading Samples — Visual Validation', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(sample_paths):
        img = Image.open(sample_paths[i])
        ax.imshow(img)
        ax.set_title(f'{sample_paths[i].name}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(_model_training_dir / 'notebooks' / 'synthetic_shading_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved validation grid to: synthetic_shading_validation.png")

## Summary

### What was done:
1. Collected clean reference images from all sources
2. Applied 3 types of synthetic shadows (polygon, gradient, radial)
3. Randomized shadow darkness (20-60%), size, position, and blur
4. Generated 350 synthetic shading images
5. Saved to `ai-service/model-training/dataset_raw/synthetic_shading/shading/`

### Disclosure
> **Shading class contains synthetically generated images (simulated shadow overlays on clean panel photos), as no source dataset provided real shading examples. This should be disclosed in the project report and ideally supplemented with real images if available before final submission.**

### Next step
Run `01_data_preprocessing.ipynb` — it will now discover `synthetic_shading/shading/` as an additional source and include it in the remapping pipeline as the `shading` class.